In [1]:
import sqlite3
import pandas as pd

df = pd.read_csv('superstore_cleaned.csv', encoding='latin-1')
conn = sqlite3.connect('superstore.db')
df.to_sql('superstore', conn, if_exists='replace', index=False)
print("Done!")

Done!


In [2]:
import sqlite3
import pandas as pd

df = pd.read_csv('superstore_cleaned.csv', encoding='latin-1')

# Fix column names - remove spaces
df.columns = df.columns.str.replace(' ', '_').str.replace('-', '_')

conn = sqlite3.connect('superstore.db')
df.to_sql('superstore', conn, if_exists='replace', index=False)

# Query function
def run_query(query):
    return pd.read_sql_query(query, conn)

print("Ready!")
print(df.columns.tolist())

Ready!
['Row_ID', 'Order_ID', 'Order_Date', 'Ship_Date', 'Ship_Mode', 'Customer_ID', 'Customer_Name', 'Segment', 'Country', 'City', 'State', 'Postal_Code', 'Region', 'Product_ID', 'Category', 'Sub_Category', 'Product_Name', 'Sales', 'Quantity', 'Discount', 'Profit', 'Order_Month']


In [3]:
q = "SELECT * FROM superstore LIMIT 2"
print(run_query(q))

   Row_ID        Order_ID  Order_Date   Ship_Date     Ship_Mode Customer_ID  \
0       1  CA-2016-152156  2016-11-08  2016-11-11  Second Class    CG-12520   
1       2  CA-2016-152156  2016-11-08  2016-11-11  Second Class    CG-12520   

  Customer_Name   Segment        Country       City  ... Region  \
0   Claire Gute  Consumer  United States  Henderson  ...  South   
1   Claire Gute  Consumer  United States  Henderson  ...  South   

        Product_ID   Category Sub_Category  \
0  FUR-BO-10001798  Furniture    Bookcases   
1  FUR-CH-10000454  Furniture       Chairs   

                                        Product_Name   Sales Quantity  \
0                  Bush Somerset Collection Bookcase  261.96        2   
1  Hon Deluxe Fabric Upholstered Stacking Chairs,...  731.94        3   

   Discount    Profit  Order_Month  
0       0.0   41.9136      2016-11  
1       0.0  219.5820      2016-11  

[2 rows x 22 columns]


In [4]:
q = "SELECT * FROM superstore LIMIT 5"
print(run_query(q))

   Row_ID        Order_ID  Order_Date   Ship_Date       Ship_Mode Customer_ID  \
0       1  CA-2016-152156  2016-11-08  2016-11-11    Second Class    CG-12520   
1       2  CA-2016-152156  2016-11-08  2016-11-11    Second Class    CG-12520   
2       3  CA-2016-138688  2016-06-12  2016-06-16    Second Class    DV-13045   
3       4  US-2015-108966  2015-10-11  2015-10-18  Standard Class    SO-20335   
4       5  US-2015-108966  2015-10-11  2015-10-18  Standard Class    SO-20335   

     Customer_Name    Segment        Country             City  ... Region  \
0      Claire Gute   Consumer  United States        Henderson  ...  South   
1      Claire Gute   Consumer  United States        Henderson  ...  South   
2  Darrin Van Huff  Corporate  United States      Los Angeles  ...   West   
3   Sean O'Donnell   Consumer  United States  Fort Lauderdale  ...  South   
4   Sean O'Donnell   Consumer  United States  Fort Lauderdale  ...  South   

        Product_ID         Category Sub_Category  

In [5]:
def run_query(query):
    return pd.read_sql_query(query, conn)

# Sales and Profit by Region

In [6]:
q1 = """
SELECT Region, 
ROUND(SUM(Sales),2) as Total_Sales,
ROUND(SUM(Profit),2) as Total_Profit
FROM superstore
GROUP BY Region
ORDER BY Total_Sales DESC
"""
print(run_query(q1))

    Region  Total_Sales  Total_Profit
0     West    725457.82     108418.45
1     East    678781.24      91522.78
2  Central    501239.89      39706.36
3    South    391721.91      46749.43


# Top 10 Most Profitable Products

In [7]:
q2 = """
SELECT Product_Name, 
ROUND(SUM(Profit),2) as Total_Profit
FROM superstore
GROUP BY Product_Name
ORDER BY Total_Profit DESC
LIMIT 10
"""
print(run_query(q2))

                                        Product_Name  Total_Profit
0              Canon imageCLASS 2200 Advanced Copier      25199.93
1  Fellowes PB500 Electric Punch Plastic Comb Bin...       7753.04
2               Hewlett Packard LaserJet 3310 Copier       6983.88
3                 Canon PC1060 Personal Laser Copier       4570.93
4  HP Designjet T520 Inkjet Large Format Printer ...       4094.98
5                  Ativa V4110MDD Micro-Cut Shredder       3772.95
6   3D Systems Cube Printer, 2nd Generation, Magenta       3717.97
7  Plantronics Savi W720 Multi-Device Wireless He...       3696.28
8               Ibico EPK-21 Electric Binding System       3345.28
9                  Zebra ZM400 Thermal Label Printer       3343.54


#  Orders Where Discount is More Than 30%

In [8]:
q3 = """
SELECT Order_ID, Product_Name, Discount, Profit
FROM superstore
WHERE Discount > 0.3
ORDER BY Discount DESC
"""
print(run_query(q3))

            Order_ID                                       Product_Name  \
0     US-2015-118983  Holmes Replacement Filter for HEPA Air Cleaner...   
1     US-2015-118983   Storex DuraTech Recycled Plastic Frosted Binders   
2     US-2017-118038                                    Economy Binders   
3     CA-2016-158568      Avery Hidden Tab Dividers for Binding Systems   
4     CA-2014-139892       Kensington 7 Outlet MasterPiece Power Center   
...              ...                                                ...   
1161  CA-2015-135251  O'Sullivan Manor Hill 2-Door Library in Briann...   
1162  CA-2017-137449  O'Sullivan Plantations 2-Door Library in Landv...   
1163  CA-2015-130183  Atlantic Metals Mobile 5-Shelf Bookcases, Cust...   
1164  CA-2017-144491  Atlantic Metals Mobile 5-Shelf Bookcases, Cust...   
1165  CA-2015-168088  Bush Heritage Pine Collection 5-Shelf Bookcase...   

      Discount    Profit  
0         0.80 -123.8580  
1         0.80   -3.8160  
2         0.80   -

# Categories Where Average Profit is Negative

In [12]:
q4 = """
SELECT 'Sub-Category',
ROUND(AVG(Profit),2) as Avg_Profit
FROM superstore
GROUP BY Category
HAVING AVG(Profit) < 0
"""
print(run_query(q4))

Empty DataFrame
Columns: ['Sub-Category', Avg_Profit]
Index: []


#  Total Orders and Revenue Per Year

In [19]:
q5_detail = """
SELECT strftime('%Y', Order_Date) as Year,
COUNT(DISTINCT Order_ID) as Total_Orders,
ROUND(SUM(Sales),2) as Total_Revenue,
ROUND(SUM(Profit),2) as Total_Profit
FROM superstore
GROUP BY Year
ORDER BY Year
"""
print(run_query(q5_detail))

   Year  Total_Orders  Total_Revenue  Total_Profit
0  2014           969      484247.50      49543.97
1  2015          1038      470532.51      61618.60
2  2016          1315      609205.60      81795.17
3  2017          1687      733215.26      93439.27


# Products With Profit Above Average

In [14]:
q6 = """
SELECT Product_Name, 
ROUND(SUM(Profit),2) as Total_Profit
FROM superstore
GROUP BY Product_Name
HAVING SUM(Profit) > (SELECT AVG(Profit) FROM superstore)
ORDER BY Total_Profit DESC
"""
print(run_query(q6))

                                           Product_Name  Total_Profit
0                 Canon imageCLASS 2200 Advanced Copier      25199.93
1     Fellowes PB500 Electric Punch Plastic Comb Bin...       7753.04
2                  Hewlett Packard LaserJet 3310 Copier       6983.88
3                    Canon PC1060 Personal Laser Copier       4570.93
4     HP Designjet T520 Inkjet Large Format Printer ...       4094.98
...                                                 ...           ...
1105   Park Ridge Embossed Executive Business Envelopes         29.27
1106    Avery Heavy-Duty EZD  Binder With Locking Rings         29.24
1107                         Sabrent 4-Port USB 2.0 Hub         29.20
1108               Belkin 6 Outlet Metallic Surge Strip         28.75
1109                                         Xerox 1932         28.71

[1110 rows x 2 columns]


#  Label Each Order as Profit or Loss

In [15]:
q7 = """
SELECT Order_ID, 
Product_Name, 
ROUND(Profit,2) as Profit,
CASE 
    WHEN Profit > 0 THEN 'Profitable'
    WHEN Profit < 0 THEN 'Loss'
    ELSE 'Breakeven'
END as Status
FROM superstore
ORDER BY Profit DESC
"""
print(run_query(q7))

            Order_ID                                      Product_Name  \
0     CA-2016-118689             Canon imageCLASS 2200 Advanced Copier   
1     CA-2017-140151             Canon imageCLASS 2200 Advanced Copier   
2     CA-2017-166709             Canon imageCLASS 2200 Advanced Copier   
3     CA-2016-117121  GBC Ibimaster 500 Manual ProClick Binding System   
4     CA-2014-116904              Ibico EPK-21 Electric Binding System   
...              ...                                               ...   
9989  US-2017-122714              Ibico EPK-21 Electric Binding System   
9990  CA-2017-134845         Lexmark MX611dhe Monochrome Laser Printer   
9991  CA-2014-169019         GBC DocuBind P400 Electric Binding System   
9992  US-2017-168116         Cubify CubeX 3D Printer Triple Head Print   
9993  CA-2016-108196         Cubify CubeX 3D Printer Double Head Print   

       Profit      Status  
0     8399.98  Profitable  
1     6719.98  Profitable  
2     5039.99  Profitable  

In [20]:
q7_summary = """
SELECT Status, COUNT(*) as Count,
ROUND(SUM(Profit),2) as Total_Profit
FROM (
    SELECT Profit,
    CASE 
        WHEN Profit > 0 THEN 'Profitable'
        WHEN Profit < 0 THEN 'Loss'
        ELSE 'Breakeven'
    END as Status
    FROM superstore
)
GROUP BY Status
"""
print(run_query(q7_summary))

       Status  Count  Total_Profit
0   Breakeven     65          0.00
1        Loss   1871    -156131.29
2  Profitable   8058     442528.31


#  CTE: Regions With Above Average Sales

In [16]:
q8 = """
WITH Regional_Sales AS (
    SELECT Region,
    ROUND(SUM(Sales),2) as Total_Sales
    FROM superstore
    GROUP BY Region
)
SELECT Region, Total_Sales
FROM Regional_Sales
WHERE Total_Sales > (SELECT AVG(Total_Sales) FROM Regional_Sales)
ORDER BY Total_Sales DESC
"""
print(run_query(q8))

  Region  Total_Sales
0   West    725457.82
1   East    678781.24


# Window Function: Rank Products by Profit Within Category

In [17]:
q9 = """
SELECT Category,
Product_Name,
ROUND(SUM(Profit),2) as Total_Profit,
RANK() OVER (PARTITION BY Category ORDER BY SUM(Profit) DESC) as Profit_Rank
FROM superstore
GROUP BY Category, Product_Name
ORDER BY Category, Profit_Rank
"""
print(run_query(q9))

        Category                                       Product_Name  \
0      Furniture  Hon Deluxe Fabric Upholstered Stacking Chairs,...   
1      Furniture            Global Deluxe High-Back Manager's Chair   
2      Furniture                         Hon Pagoda Stacking Chairs   
3      Furniture  Hon 4070 Series Pagoda Armless Upholstered Sta...   
4      Furniture  Office Star - Professional Matrix Back Chair w...   
...          ...                                                ...   
1845  Technology  Epson TM-T88V Direct Thermal Printer - Monochr...   
1846  Technology  Cisco TelePresence System EX90 Videoconferenci...   
1847  Technology          Cubify CubeX 3D Printer Triple Head Print   
1848  Technology          Lexmark MX611dhe Monochrome Laser Printer   
1849  Technology          Cubify CubeX 3D Printer Double Head Print   

      Total_Profit  Profit_Rank  
0          1927.44            1  
1          1558.59            2  
2          1540.70            3  
3          

# Window Function: Running Total of Sales by Month

In [18]:
q10 = """
SELECT strftime('%Y-%m', Order_Date) as Month,
ROUND(SUM(Sales),2) as Monthly_Sales,
ROUND(SUM(SUM(Sales)) OVER (ORDER BY strftime('%Y-%m', Order_Date)),2) as Running_Total
FROM superstore
GROUP BY Month
ORDER BY Month
"""
print(run_query(q10))

      Month  Monthly_Sales  Running_Total
0   2014-01       14236.90       14236.90
1   2014-02        4519.89       18756.79
2   2014-03       55691.01       74447.80
3   2014-04       28295.35      102743.14
4   2014-05       23648.29      126391.43
5   2014-06       34595.13      160986.56
6   2014-07       33946.39      194932.95
7   2014-08       27909.47      222842.42
8   2014-09       81777.35      304619.77
9   2014-10       31453.39      336073.16
10  2014-11       78628.72      414701.88
11  2014-12       69545.62      484247.50
12  2015-01       18174.08      502421.57
13  2015-02       11951.41      514372.98
14  2015-03       38726.25      553099.24
15  2015-04       34195.21      587294.45
16  2015-05       30131.69      617426.13
17  2015-06       24797.29      642223.42
18  2015-07       28765.33      670988.75
19  2015-08       36898.33      707887.08
20  2015-09       64595.92      772483.00
21  2015-10       31404.92      803887.92
22  2015-11       75972.56      87